In [0]:
%pip install -U \
    category-encoders \
    cloudpickle==3.0.0 \
    nltk==3.9.2 \
    defusedxml==0.7.1 \
    graphviz==0.20.3 \
    holidays==0.54 \
    lightgbm==4.5.0 \
    matplotlib==3.9.2 \
    psutil==5.9.8 \
    pyarrow==15.0.2 \
    optuna \
    plotly \
    sentence-transformers \
    scikit-learn \
    torch \
    tqdm


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install emoji nltk keybert

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install hdbscan umap

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
train_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/train.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)


In [0]:
import pandas as pd
test_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/test.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)

In [0]:
df=pd.concat([train_df,test_df],axis=0)

In [0]:
df.isnull().sum()

customer_identifier     0
medicine_name           0
rating                  0
effectiveness           0
side_effects            0
illness                 1
review_benefits        23
review_sideEffects     98
review_overall         13
dtype: int64

In [0]:
mode_illness = df['illness'].mode()[0]
df['illness'] = df['illness'].fillna(mode_illness)
df['review_benefits'] = df['review_benefits'].fillna("")
df['review_sideEffects'] = df['review_sideEffects'].fillna("")
df['review_overall'] = df['review_overall'].fillna("")
df['rating']=df['rating'].astype(str)

In [0]:
review_template = (
    "review_benefits: {review_benefits}\n"
    "review_sideEffects: {review_sideEffects}\n"
    "review_overall: {review_overall}  \n"
    "medicine_name: {medicine_name}  \n"
    "effectiveness: {effectiveness}  \n"
    "side_effects: {side_effects}  \n"
    "rating: {rating}  \n"
    "illness: {illness}  "
)

df['combined_feats'] = df.apply(lambda row: review_template.format(**row), axis=1)


In [0]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')
text_embeddings = model.encode(df["combined_feats"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

In [0]:
display(df)

customer_identifier,medicine_name,rating,effectiveness,side_effects,illness,review_benefits,review_sideEffects,review_overall,combined_feats
2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure alone or with other agents in the managment of hypertension mangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid","review_benefits: slowed the progression of left ventricular dysfunction into overt heart failure alone or with other agents in the managment of hypertension mangagement of congestive heart failur review_sideEffects: cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness review_overall: monitor blood pressure , weight and asses for resolution of fluid medicine_name: enalapril effectiveness: Highly Effective side_effects: Mild Side Effects rating: 4 illness: management of congestive heart failure"
3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,"Although this type of birth control has more cons than pros, it did help with my cramps. It's also effective with the prevention of pregnancy. (Along with use of condoms as well)","Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results.","I Hate This Birth Control, I Would Not Suggest This To Anyone.","review_benefits: Although this type of birth control has more cons than pros, it did help with my cramps. It's also effective with the prevention of pregnancy. (Along with use of condoms as well) review_sideEffects: Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results. review_overall: I Hate This Birth Control, I Would Not Suggest This To Anyone. medicine_name: ortho-tri-cyclen effectiveness: Highly Effective side_effects: Severe Side Effects rating: 1 illness: birth prevention"
1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,"I was used to having cramps so badly that they would leave me balled up in bed for at least 2 days. The Ponstel doesn't take the pain away completely, but takes the edge off so much that normal activities were possible. Definitely a miracle medication!!",Heavier bleeding and clotting than normal.,"I took 2 pills at the onset of my menstrual cramps and then every 8-12 hours took 1 pill as needed for about 3-4 days until cramps were over. If cramps are bad, make sure to take every 8 hours on the dot because the medication stops working suddenly and unfortunately takes about an hour to an hour and a half to kick back in.. if cramps are only moderate, taking every 12 hours is okay.","review_benefits: I was used to having cramps so badly that they would leave me balled up in bed for at least 2 days. The Ponstel doesn't take the pain away completely, but takes the edge off so much that normal activities were possible. Definitely a miracle medication!! review_sideEffects: Heavier bleeding and clotting than normal. review_overall: I took 2 pills at the onset of my menstrual cramps and then every 8-12 hours took 1 pill as needed for about 3-4 days until cramps were over. If cramps are bad, make sure to take every 8 hours on the dot because the medication stops working suddenly and unfortunately takes about an hour to an h

In [0]:
from sklearn.decomposition import PCA
import plotly.express as px
import numpy as np



pca = PCA(n_components=2)
X_vis = pca.fit_transform(text_embeddings)

fig = px.scatter(
    x=X_vis[:, 0],
    y=X_vis[:, 1],
    # color=df["effectiveness"].astype(str),
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2', 'color': 'Rating'},
    title="PCA Visualization of Reviews (Sentence Embeddings + Rating)"
)
fig.show()

In [0]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
from sklearn.decomposition import PCA

# Find optimal k using silhouette score
scores = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_reduced)
    score = silhouette_score(text_embeddings, kmeans.labels_)
    scores.append(score)

fig = px.line(
    x=list(range(2, 10)),
    y=scores,
    labels={"x": "Number of Clusters (k)", "y": "Silhouette Score"},
    title="Silhouette Score vs Number of Clusters (k)"
)
fig.show()

# Fit final model
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42)
df["cohort"] = kmeans.fit_predict(text_embeddings)

In [0]:
from sklearn.decomposition import PCA
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

numeric_features = df[["rating"]].values
scaled_num = StandardScaler().fit_transform(numeric_features)
features_for_pca = text_embeddings

pca = PCA(n_components=2)
X_vis = pca.fit_transform(features_for_pca)

# Compute centroids in PCA space
kmeans = KMeans(n_clusters=df["cohort"].nunique(), random_state=42)
kmeans.fit(features_for_pca)
centroids_pca = pca.transform(kmeans.cluster_centers_)

fig = px.scatter(
    x=X_vis[:, 0],
    y=X_vis[:, 1],
    color=df["cohort"].astype(str),
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2', 'color': 'Cohort'},
    title="PCA Visualization of Reviews (Sentence Embeddings + Cohort)"
)
fig.add_scatter(
    x=centroids_pca[:, 0],
    y=centroids_pca[:, 1],
    mode='markers',
    marker=dict(symbol='x', size=15, color='black'),
    name='Centroid'
)
fig.show()

In [0]:
display(df)

customer_identifier,medicine_name,rating,effectiveness,side_effects,illness,review_benefits,review_sideEffects,review_overall,combined_feats,cohort,review_overall_cleaned,review_sideEffects_cleaned,review_benefits_cleaned
2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dysfunction into overt heart failure alone or with other agents in the managment of hypertension mangagement of congestive heart failur,"cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness","monitor blood pressure , weight and asses for resolution of fluid","review_benefits: slowed the progression of left ventricular dysfunction into overt heart failure alone or with other agents in the managment of hypertension mangagement of congestive heart failur review_sideEffects: cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness review_overall: monitor blood pressure , weight and asses for resolution of fluid medicine_name: enalapril effectiveness: Highly Effective side_effects: Mild Side Effects rating: 4 illness: management of congestive heart failure",1,monitor blood pressure weight ass resolution fluid,cough hypotension proteinuria impotence renal failure angina pectoris tachycardia eosinophilic pneumonitis taste disturbance anusease anorecia weakness fatigue insominca weakness,slowed progression left ventricular dysfunction overt heart failure alone agent managment hypertension mangagement congestive heart failur
3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,"Although this type of birth control has more cons than pros, it did help with my cramps. It's also effective with the prevention of pregnancy. (Along with use of condoms as well)","Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results.","I Hate This Birth Control, I Would Not Suggest This To Anyone.","review_benefits: Although this type of birth control has more cons than pros, it did help with my cramps. It's also effective with the prevention of pregnancy. (Along with use of condoms as well) review_sideEffects: Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results. review_overall: I Hate This Birth Control, I Would Not Suggest This To Anyone. medicine_name: ortho-tri-cyclen effectiveness: Highly Effective side_effects: Severe Side Effects rating: 1 illness: birth prevention",1,hate birth control would suggest anyone,heavy cycle cramp hot flash fatigue lasting cycle month concidering changing different bc first kind bc unfortunately due constant hassel happy result,although type birth control con pro help cramp also effective prevention pregnancy along condom well
1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,"I was used to having cramps so badly that they would leave me balled up in bed for at least 2 days. The Ponstel doesn't take the pain away completely, but takes the edge off so much that normal activities were possible. Definitely a miracle medication!!",Heavier bleeding and clotting than normal.,"I took 2 pills at the onset of my menstrual cramps and then every 8-12 hours took 1 pill as needed for about 3-4 days until cramps were over. If cramps are bad, make sure to take every 8 hours on the dot because the medication stops working suddenly and unfortunately takes about an hour t

In [0]:
df['rating']=df['rating'].astype(int)

In [0]:
cluster_summary = (
    df.groupby("cohort")
      .agg({
          "rating": ["mean", "count"],
          "medicine_name": lambda x: x.value_counts().index[0],
          "illness": lambda x: x.value_counts().index[0]
      })
)
cluster_summary.columns = ["avg_rating", "count", "top_medicine", "top_illness"]
display(cluster_summary)

avg_rating,count,top_medicine,top_illness
7.120967741935484,620,retin-a,acne
6.749013806706114,2028,lipitor,birth control
7.141806020066889,1495,lexapro,depression


In [0]:
cluster_summary.to_dict()

{'avg_rating': {0: 7.120967741935484,
  1: 6.749013806706114,
  2: 7.141806020066889},
 'count': {0: 620, 1: 2028, 2: 1495},
 'top_medicine': {0: 'retin-a', 1: 'lipitor', 2: 'lexapro'},
 'top_illness': {0: 'acne', 1: 'birth control', 2: 'depression'}}

In [0]:
import re, nltk ,emoji
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
additional_stopwords = {
    "drug", "medicine", "tablet", "doctor", "patient", "review_benefits", "review_sideEffects","review_sideeffects","review_benefits"
    "review_overall", "illness","medicine_name","symptom","reflux","side_effects"
"mg", "day", "night","pill",
 "take", "took", "used", "taking", "pill", "one", "review",
    "dose", "dosage", "medication", "treatment", "therapy", "prescribed", "prescription", 
    "prescribe", "physician", "med", "antibiotic",  "application", "apply", "use", 
    "using", "take", "taken", "stop", "stopped", "started", "starting", "start", "continue",
    "continued", "course", "treat", "treated", "treating", "dos", "mcg", "tab", "daily", 
    "nightly", "bedtime", "morning", "evening", "hour", "per", "pm", "qd", "weekly", "month",
    "week", "year", "night", "time", "two", "three", "four", "five", "every", "twice", "long",
    "within", "since", "around", "last", "next", "ago", "short","month",
    "january", "february", "march", "april", "may", "june", "july", "august", "september",
    "october", "november", "december", "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep",
    "sept", "oct", "nov", "dec","effect", "pill", "tablet"
}
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english')).union(additional_stopwords)
lemmatizer = WordNetLemmatizer()

# ----------------------------------------------------------------
# 🔹 3. Text Preprocessor Function
# ----------------------------------------------------------------
def preprocess_text_fast(text):
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, language='en')
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)
df["review_overall_cleaned"] = df["review_overall"].apply(preprocess_text_fast)
df["review_sideEffects_cleaned"] = df["review_sideEffects"].apply(preprocess_text_fast)
df["review_benefits_cleaned"] = df["review_benefits"].apply(preprocess_text_fast)


[nltk_data] Downloading package stopwords to /home/spark-
[nltk_data]     dba71c9a-3145-4b6c-adb0-3e/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/spark-
[nltk_data]     dba71c9a-3145-4b6c-adb0-3e/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)
X_tfidf = vectorizer.fit_transform(df["review_overall_cleaned"])

terms = np.array(vectorizer.get_feature_names_out())
cluster_keywords = {}

for c in sorted(df["cohort"].unique()):
    mask = (df["cohort"] == c).values  # Convert to numpy array
    cluster_docs = X_tfidf[mask]
    mean_tfidf = np.asarray(cluster_docs.mean(axis=0)).ravel()
    top_idx = mean_tfidf.argsort()[-10:][::-1]
    cluster_keywords[c] = terms[top_idx]

for c, words in cluster_keywords.items():
    print(f"Cluster {c}: {', '.join(words)}")

Cluster 0: skin, face, acne, cream, applied, retin, area, dry, month, product
Cluster 1: day, pain, time, effect, week, hour, pill, year, tablet, symptom
Cluster 2: depression, year, sleep, month, anxiety, effect, needed, increased, feel, week


In [0]:
df.columns

Index(['customer_identifier', 'medicine_name', 'rating', 'effectiveness',
       'side_effects', 'illness', 'review_benefits', 'review_sideEffects',
       'review_overall', 'combined_feats', 'cohort', 'review_overall_cleaned',
       'review_sideEffects_cleaned', 'review_benefits_cleaned'],
      dtype='object')

In [0]:
from keybert import KeyBERT
kw_model = KeyBERT()

for c in sorted(df["cohort"].unique()):
    print(f"\nCluster {c}:")
    # review_overall_cleaned
    text_overall = " ".join(df.loc[df["cohort"] == c, "illness"].tolist())
    keywords_overall = kw_model.extract_keywords(text_overall, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  illness keywords: {[k[0] for k in keywords_overall]}")

    text_overall = " ".join(df.loc[df["cohort"] == c, "medicine_name"].tolist())
    keywords_overall = kw_model.extract_keywords(text_overall, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  medicine_name keywords: {[k[0] for k in keywords_overall]}")

    text_overall = " ".join(df.loc[df["cohort"] == c, "review_overall_cleaned"].tolist())
    keywords_overall = kw_model.extract_keywords(text_overall, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  review_overall_cleaned keywords: {[k[0] for k in keywords_overall]}")
    # review_sideEffects_cleaned
    text_side = " ".join(df.loc[df["cohort"] == c, "review_sideEffects_cleaned"].tolist())
    keywords_side = kw_model.extract_keywords(text_side, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  review_sideEffects_cleaned keywords: {[k[0] for k in keywords_side]}")
    # review_benefits_cleaned
    text_benefits = " ".join(df.loc[df["cohort"] == c, "review_benefits_cleaned"].tolist())
    keywords_benefits = kw_model.extract_keywords(text_benefits, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  review_benefits_cleaned keywords: {[k[0] for k in keywords_benefits]}")
    # Print sample combined_feats
    text_benefits = " ".join(df.loc[df["cohort"] == c, "combined_feats"].tolist())
    keywords_benefits = kw_model.extract_keywords(text_benefits, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    print(f"  combined_feats keywords: {[k[0] for k in keywords_benefits]}")


Cluster 0:
  illness keywords: ['eyelashes acne', 'lashes acne', 'acne whiteheads', 'eyes acne', 'acne sparse', 'acne cosmetic', 'acne scars', 'acne pimples', 'lines acne', 'eye acne']
  medicine_name keywords: ['tetracycline retin', 'zovirax retin', 'tazorac retin', 'retin zovirax', 'minocycline retin', 'renova cleocin', 'differin zovirax', 'azelex retin', 'doxycycline retin', 'retin tazorac']
  review_overall_cleaned keywords: ['applied acne', 'applied dermatitis', 'existing acne', 'skincare regimen', 'acne patch', 'acne applied', 'healing acne', 'acne return', 'topical acne', 'reduction acne']
  review_sideEffects_cleaned keywords: ['skin condition', 'moisturizer redness', 'skin symptom', 'itchiness redness', 'hyperpigmentation skin', 'redness itchiness', 'dermatitis flare', 'skin problem', 'skin redness', 'improvement skin']
  review_benefits_cleaned keywords: ['cosmetic procedure', 'skin improvement', 'improvement skin', 'makeup cured', 'acne topicals', 'improvement acne', 'resto